# Create ARAMIS SERI Awards (Swiss State Secretariat for Education, Research and Innovation)

Creates awards for the **State Secretariat for Education, Research and Innovation (SERI / SBFI)**
from **ARAMIS**, the Swiss federal administration's research project database, using SBFI's own
open-data export on opendata.swiss (CKAN dataset `aramis`, terms_open):
`https://datenausgabe.aramis.admin.ch/full_export.json` (~48,000 projects, all federal offices).

**Shared portal, routed by funding office (runbook §2.3.2).** ARAMIS covers ~60 federal research
units (Innosuisse 17.8k, BFE 6.4k, BAFU, BAG ...). Only rows whose `research_unit_short_name` is
SERI are ingested here:
- `SBFI` -- SERI itself (EU framework-programme participation it funds directly, Eurostars/AAL,
  vocational-education research, evaluations, space projects ...)
- `SBF` -- Staatssekretariat für Bildung und Forschung, SERI's legal predecessor (renamed SBFI on
  2013-01-01; no separate OpenAlex/Crossref funder row) -- 37 grant-level rows
- `COST` -- Swiss COST-action participation funded by SBF/SBFI (numbers C05.xxxx-C16.xxxx).
  C9x-C04 numbers are BBW era (F4320326458) and are NOT included.

Not included (other funders): EU FRP 1994-2004 (BBW, F4320326458), BBT (F4320326462),
Innosuisse and all other offices.

**Scope filter (grant-level only, applied in the script):** ARAMIS also records SERI's budget
allocations -- the federal block grant to the SNSF, Horizon Europe / EURATOM mandatory
contributions, CERN/ESO/ESA/EMBL/ESRF/ILL membership, core subsidies to academies and Art. 15
research institutions, Leading House mandates. 752 such rows (CHF 37.5bn) are not awards to a
project/grantee and are excluded (listed in the script's excluded_allocations.csv). In the
institutional sections only rows with a project-level number are kept.

**Fields:** `funder_award_id` = ARAMIS project number as published (e.g. `15.0262`, `22.00187`,
`C14.0094`, `P2016-Eurostars-011`) -- the SERI contract number papers cite (173 of 466 distinct
SERI citation stubs collapse; MB22.xxxxx Horizon-Europe guarantee numbers and ESKAS scholarships
are not in ARAMIS). `amount` = `granted_total_costs` in **CHF** (SERI's granted amount; for
EU-FP rows the SERI funding of the Swiss partner). Title: English if present, else German /
French / Italian. PI: ARAMIS's contact block (name + institution only; e-mail/address dropped),
blank when the contact is a SERI officer (~13% PI coverage, mostly COST). No stable public project
URL is in the export (ARAMIS pages are keyed by an internal ProjectID), so `landing_page_url` is NULL.

**Prerequisites:** run `scripts/local/aramis_seri_to_s3.py` (uploads
`s3://openalex-ingest/awards/aramis_seri/aramis_seri_projects.parquet`, §1.4 shrink guard).

**Funder (Path A, F4320*):** funder_id `4320335238` (Staatssekretariat für Bildung, Forschung und
Innovation, ror 01kw63t33, doi 10.13039/501100007352), read from `openalex.funders.funders`.

**Priority:** `533` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aramis_seri_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/aramis_seri/aramis_seri_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects, COUNT(DISTINCT aramis_number) as distinct_numbers
FROM openalex.awards.aramis_seri_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.aramis_seri_raw;

In [ ]:
%sql
SELECT research_unit, section, COUNT(*) AS n, ROUND(SUM(TRY_CAST(granted_total_costs_chf AS DOUBLE)), 0) AS chf
FROM openalex.awards.aramis_seri_raw
GROUP BY 1, 2 ORDER BY n DESC;

## Step 1.6: Funder existence check (Path A)
F4320335238 must resolve to exactly one row in `openalex.funders.funders`; if not, STOP.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320335238;

## Step 2: Create SERI Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.aramis_seri_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320335238  -- State Secretariat for Education, Research and Innovation
),
g AS (
    SELECT
        TRIM(aramis_number) AS funder_award_id,
        COALESCE(NULLIF(TRIM(title_en), ''), NULLIF(TRIM(title_de), ''), NULLIF(TRIM(title_fr), ''), NULLIF(TRIM(title_it), '')) AS title,
        NULLIF(TRIM(abstract), '') AS abstract,
        TRY_CAST(granted_total_costs_chf AS DOUBLE) AS amount,
        NULLIF(TRIM(section), '') AS section,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS start_d,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS end_d,
        CASE WHEN lead_family_name IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(lead_institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead
    FROM openalex.awards.aramis_seri_raw
    WHERE aramis_number IS NOT NULL AND TRIM(aramis_number) != ''
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
    g.title as display_name,
    g.abstract as description,
    f.funder_id,
    g.funder_award_id,
    g.amount,
    'CHF' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    g.section as funder_scheme,
    'aramis_seri' as provenance,
    g.start_d as start_date,
    g.end_d as end_date,
    YEAR(g.start_d) as start_year,
    YEAR(g.end_d) as end_year,
    g.lead as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    CAST(NULL AS STRING) as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aramis_seri' AND priority = 533;

-- Insert into openalex_awards_raw with priority 533 (direct funder ingest; higher wins
-- under the 2026-06-20 DESC dedup, so it outranks the priority-0 acknowledgement shells).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    533 as priority
FROM openalex.awards.aramis_seri_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.aramis_seri_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.aramis_seri_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_chf
FROM openalex.awards.aramis_seri_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_chf
FROM openalex.awards.aramis_seri_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency checks: PI and title top-20 must be a real long tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.aramis_seri_awards
WHERE lead_investigator IS NOT NULL
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.aramis_seri_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(lead_investigator) as has_pi
FROM openalex.awards.aramis_seri_awards;

In [ ]:
%sql
-- Shape check (2.1.1): ARAMIS numbers only, never a blank/system id.
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^[0-9]{2}\\.[0-9]{4,5}' THEN 1 ELSE 0 END) AS contract_numbers,
    SUM(CASE WHEN funder_award_id RLIKE '^C[0-9]{2}\\.[0-9]{4}$' THEN 1 ELSE 0 END) AS cost_numbers,
    SUM(CASE WHEN funder_award_id RLIKE '^P20[0-9]{2}-(Eurostars|AAL)-' THEN 1 ELSE 0 END) AS eurostars_aal,
    SUM(CASE WHEN funder_award_id IS NULL OR TRIM(funder_award_id) = '' THEN 1 ELSE 0 END) AS blank_ids,
    COUNT(*) AS total
FROM openalex.awards.aramis_seri_awards;

In [ ]:
%sql
-- Citation stubs (priority 0) under F4320335238 that collapse onto this ingest.
SELECT c.provenance, COUNT(*) as stub_rows, COUNT(t.id) as collapse_onto_aramis
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.aramis_seri_awards t ON t.id = c.id
WHERE c.funder_id = 4320335238 AND c.priority < 3
GROUP BY c.provenance;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'aramis_seri'
GROUP BY provenance, priority;